In [ ]:
# Edit only attached Kaggle Input paths and reduce batch sizes only for OOM.
from pathlib import Path

PUBLIC_LEGALIR_PATH = Path(
    "/kaggle/input/<public-legalir-dataset>/public-official.json"
)
CORPUS_PATH = Path(
    "/kaggle/input/datasets/mduy2911/legalir/selected-contexts"
)
DENSE_MODEL_PATH = Path(
    "/kaggle/input/datasets/mduy2911/bge-m3-kaggle"
)
RERANKER_MODEL_PATH = Path(
    "/kaggle/input/datasets/mduy2911/bge-reranker-v2-m3-kaggle"
)

CORPUS_BATCH_SIZE = 256
QUERY_BATCH_SIZE = 64
RERANKER_BATCH_SIZE = 128

DENSE_MODEL_NAME = "BAAI/bge-m3"
DENSE_DECLARED_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANKER_DECLARED_REVISION = "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"
EXPECTED_PUBLIC_SAMPLES = 1_000
EXPECTED_DOCUMENTS = 8_532
EXPECTED_CHUNKS = 199_816
CHUNK_SIZE = 2_000
CHUNK_OVERLAP = 200
DENSE_MAX_LENGTH = 8_192
TOP_K_CHUNKS = 2_000
DOCUMENT_AGGREGATION = "sum_top_2"
DENSE_AGGREGATION_CHUNKS = 2
CANDIDATE_DEPTH = 100
SUPPORT_POOL_SIZE = 8
CE_SELECTED_CHUNKS = 2
RERANKER_MAX_SEQUENCE_LENGTH = 8_192
FINAL_K = 5

OUTPUT_PATH = Path("/kaggle/working/submission.json")
METADATA_PATH = Path(
    "/kaggle/working/legalir_dense_cross_encoder_public_inference_metadata.json"
)


In [ ]:
# Enforce local-only execution before loading either model.
import os

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for path, description, must_be_directory in (
    (PUBLIC_LEGALIR_PATH, "official public LegalIR input", False),
    (CORPUS_PATH, "LegalIR corpus directory", True),
    (DENSE_MODEL_PATH, "complete local BGE-M3 snapshot", True),
    (RERANKER_MODEL_PATH, "complete local BGE reranker snapshot", True),
):
    exists = path.is_dir() if must_be_directory else path.is_file()
    if not exists:
        raise FileNotFoundError(f"Attach the {description} at: {path}")

for name, value in (
    ("CORPUS_BATCH_SIZE", CORPUS_BATCH_SIZE),
    ("QUERY_BATCH_SIZE", QUERY_BATCH_SIZE),
    ("RERANKER_BATCH_SIZE", RERANKER_BATCH_SIZE),
):
    if not isinstance(value, int) or isinstance(value, bool) or value <= 0:
        raise ValueError(f"{name} must be a positive integer")

assert CHUNK_SIZE == 2_000 and CHUNK_OVERLAP == 200
assert DENSE_MAX_LENGTH == 8_192 and TOP_K_CHUNKS == 2_000
assert DOCUMENT_AGGREGATION == "sum_top_2"
assert DENSE_AGGREGATION_CHUNKS == 2 and CANDIDATE_DEPTH == 100
assert SUPPORT_POOL_SIZE == 8 and CE_SELECTED_CHUNKS == 2
assert RERANKER_MAX_SEQUENCE_LENGTH == 8_192
assert FINAL_K == 5
assert OUTPUT_PATH == Path("/kaggle/working/submission.json")
assert OUTPUT_PATH != METADATA_PATH


In [ ]:
# Standalone public-input loading, source-preserving chunking, and dense retrieval.
import gc
import json
from collections import defaultdict
from math import isfinite
from time import perf_counter

import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer


def reject_duplicate_object_keys(pairs):
    value = {}
    for key, item in pairs:
        if key in value:
            raise ValueError(f"duplicate JSON object key: {key!r}")
        value[key] = item
    return value


def read_json_strict(path: Path):
    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream, object_pairs_hook=reject_duplicate_object_keys)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc


def load_public_legalir(path: Path) -> dict:
    value = read_json_strict(path)
    if not isinstance(value, dict):
        raise ValueError(f"{path}: expected a top-level object keyed by question ID")
    if len(value) != EXPECTED_PUBLIC_SAMPLES:
        raise ValueError(
            f"{path}: expected exactly {EXPECTED_PUBLIC_SAMPLES} samples, "
            f"got {len(value)}"
        )
    for sample_id, sample in value.items():
        if not isinstance(sample_id, str):
            raise TypeError("public question IDs must be JSON strings")
        if not isinstance(sample, dict) or set(sample) != {"question", "answer"}:
            raise ValueError(
                f"sample {sample_id!r}: expected exactly question and answer fields"
            )
        if not isinstance(sample["question"], str):
            raise TypeError(f"sample {sample_id!r}: question must be a string")
        if sample["answer"] is not None:
            raise ValueError(f"sample {sample_id!r}: public answer must be explicit null")
    return value


def canonical_corpus_id(raw_id, source: Path) -> str:
    if not isinstance(raw_id, (str, int)) or isinstance(raw_id, bool):
        raise TypeError(f"{source}: corpus document id must be a string or integer")
    if isinstance(raw_id, str) and not raw_id:
        raise ValueError(f"{source}: corpus document id must not be empty")
    return str(raw_id)


def load_corpus(path: Path) -> list[dict]:
    paths = sorted(
        item for item in path.rglob("*")
        if item.is_file() and item.suffix.lower() == ".json"
    )
    if not paths:
        raise ValueError(f"{path}: corpus directory contains no JSON files")
    documents = []
    seen_ids = set()
    for json_path in paths:
        value = read_json_strict(json_path)
        values = value if isinstance(value, list) else [value]
        if not all(isinstance(document, dict) for document in values):
            raise ValueError(f"{json_path}: expected document object(s)")
        for document in values:
            document_id = canonical_corpus_id(document.get("id"), json_path)
            if document_id in seen_ids:
                raise ValueError(f"duplicate corpus document ID: {document_id!r}")
            passage = document.get("passage")
            if not isinstance(passage, str):
                raise TypeError(f"document {document_id!r}: passage must be a string")
            seen_ids.add(document_id)
            documents.append({"document_id": document_id, "passage": passage})
    return documents


def chunk_corpus(documents: list[dict]) -> list[dict]:
    step = CHUNK_SIZE - CHUNK_OVERLAP
    if CHUNK_SIZE <= 0 or CHUNK_OVERLAP < 0 or step <= 0:
        raise ValueError("invalid fixed-window chunk controls")
    chunks = []
    for document in documents:
        document_id = document["document_id"]
        passage = document["passage"]
        for chunk_index, start in enumerate(range(0, len(passage), step)):
            end = min(start + CHUNK_SIZE, len(passage))
            chunks.append({
                "chunk_id": f"{document_id}:{chunk_index}",
                "document_id": document_id,
                "text": passage[start:end],
            })
            if end == len(passage):
                break
    return chunks


def model_metadata(model, model_name: str, declared_revision: str, path: Path) -> dict:
    value = getattr(model.config, "_commit_hash", None)
    config_hash = value.strip() if isinstance(value, str) and value.strip() else None
    if config_hash is None:
        revision_status = "declared-offline-snapshot"
    elif config_hash == declared_revision:
        revision_status = "verified-from-config"
    else:
        raise RuntimeError(
            f"{model_name} config _commit_hash {config_hash!r} does not match "
            f"declared revision {declared_revision!r}"
        )
    return {
        "model_name": model_name,
        "declared_revision": declared_revision,
        "config_commit_hash": config_hash,
        "revision_status": revision_status,
        "local_input_path": str(path),
    }


def load_dense_model() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle CUDA accelerator")
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL_PATH, local_files_only=True)
    model = AutoModel.from_pretrained(
        DENSE_MODEL_PATH, dtype=torch.float16, local_files_only=True
    )
    tokenizer_limit = int(tokenizer.model_max_length)
    model_limit = int(getattr(model.config, "max_position_embeddings", tokenizer_limit))
    if tokenizer_limit < DENSE_MAX_LENGTH or model_limit < DENSE_MAX_LENGTH:
        raise RuntimeError("local dense model does not support max_length=8192")
    metadata = model_metadata(
        model, DENSE_MODEL_NAME, DENSE_DECLARED_REVISION, DENSE_MODEL_PATH
    )
    model.to("cuda")
    model.eval()
    return {
        "tokenizer": tokenizer,
        "model": model,
        "metadata": metadata,
        "load_seconds": perf_counter() - started,
    }


def encode_normalized_cls(model_bundle: dict, texts: list[str], batch_size: int) -> dict:
    embeddings = []
    started = perf_counter()
    for batch_start in range(0, len(texts), batch_size):
        batch = texts[batch_start:batch_start + batch_size]
        inputs = model_bundle["tokenizer"](
            batch, padding=True, truncation=True, max_length=DENSE_MAX_LENGTH,
            return_tensors="pt",
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        with torch.no_grad():
            outputs = model_bundle["model"](**inputs, return_dict=True)
            embedding = outputs.last_hidden_state[:, 0]
            embedding = F.normalize(embedding, p=2, dim=1)
        if embedding.ndim != 2 or not torch.isfinite(embedding).all():
            raise RuntimeError("dense encoder returned invalid CLS embeddings")
        embeddings.append(embedding.cpu())
    encoded = torch.cat(embeddings, dim=0)
    if encoded.shape[0] != len(texts):
        raise RuntimeError("dense embedding count mismatch")
    return {"embeddings": encoded, "seconds": perf_counter() - started}


def aggregate_dense_hits(chunk_hits: list[dict]) -> list[dict]:
    grouped = defaultdict(list)
    for hit in chunk_hits:
        if not isfinite(hit["score"]):
            raise RuntimeError("dense retrieval produced a non-finite score")
        grouped[hit["document_id"]].append(hit)
    ranked_documents = []
    for document_id, hits in grouped.items():
        ordered_hits = sorted(
            hits, key=lambda hit: (-hit["score"], hit["chunk_rank"], hit["chunk_index"])
        )
        dense_evidence = ordered_hits[:DENSE_AGGREGATION_CHUNKS]
        support = ordered_hits[:SUPPORT_POOL_SIZE]
        ranked_documents.append({
            "document_id": document_id,
            "dense_score": sum(hit["score"] for hit in dense_evidence),
            "best_chunk_rank": min(hit["chunk_rank"] for hit in hits),
            "supporting_chunk_indices": [hit["chunk_index"] for hit in support],
            "supporting_chunk_scores": [hit["score"] for hit in support],
        })
    ranked_documents.sort(key=lambda document: (
        -document["dense_score"],
        document["best_chunk_rank"],
        document["document_id"],
    ))
    selected = ranked_documents[:CANDIDATE_DEPTH]
    if len(selected) != CANDIDATE_DEPTH:
        raise RuntimeError(f"expected {CANDIDATE_DEPTH} unique candidate documents")
    for original_rank, document in enumerate(selected, start=1):
        document["original_rank"] = original_rank
    candidate_ids = [document["document_id"] for document in selected]
    if len(candidate_ids) != len(set(candidate_ids)):
        raise RuntimeError("dense candidate ranking contains duplicate document IDs")
    return selected


def retrieve_dense(
    query_embeddings: torch.Tensor, corpus_embeddings: torch.Tensor,
    chunks: list[dict], sample_ids: list[str],
) -> dict:
    if query_embeddings.shape[0] != len(sample_ids):
        raise ValueError("query embedding count mismatch")
    torch.cuda.synchronize()
    started = perf_counter()
    passage_embeddings = corpus_embeddings.to("cuda")
    candidates_by_query = {}
    for batch_start in range(0, len(sample_ids), QUERY_BATCH_SIZE):
        batch_ids = sample_ids[batch_start:batch_start + QUERY_BATCH_SIZE]
        query_batch = query_embeddings[
            batch_start:batch_start + len(batch_ids)
        ].to("cuda")
        similarities = query_batch @ passage_embeddings.T
        if not torch.isfinite(similarities).all():
            raise RuntimeError("dense dot product produced non-finite scores")
        top_scores, top_indices = torch.topk(
            similarities, k=TOP_K_CHUNKS, dim=1, largest=True, sorted=True
        )
        for row, sample_id in enumerate(batch_ids):
            raw_hits = list(zip(
                top_scores[row].float().cpu().tolist(),
                top_indices[row].cpu().tolist(),
            ))
            raw_hits.sort(key=lambda item: (-item[0], item[1]))
            hits = []
            for rank, (score_value, index_value) in enumerate(raw_hits, start=1):
                chunk_index = int(index_value)
                hits.append({
                    "document_id": chunks[chunk_index]["document_id"],
                    "chunk_index": chunk_index,
                    "score": float(score_value),
                    "chunk_rank": rank,
                })
            candidates_by_query[sample_id] = aggregate_dense_hits(hits)
    torch.cuda.synchronize()
    seconds = perf_counter() - started
    del passage_embeddings
    torch.cuda.empty_cache()
    if set(candidates_by_query) != set(sample_ids):
        raise RuntimeError("dense retrieval did not return every public query")
    return {"candidates": candidates_by_query, "seconds": seconds}


In [ ]:
# Frozen cross-encoder scoring and strict LegalIR prediction preflight.
def load_reranker() -> dict:
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(
        RERANKER_MODEL_PATH, local_files_only=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL_PATH, dtype=torch.float16, local_files_only=True
    )
    tokenizer_limit = int(tokenizer.model_max_length)
    model_limit = int(getattr(model.config, "max_position_embeddings", tokenizer_limit))
    if (
        tokenizer_limit < RERANKER_MAX_SEQUENCE_LENGTH
        or model_limit < RERANKER_MAX_SEQUENCE_LENGTH
    ):
        raise RuntimeError("local reranker does not support max sequence length 8192")
    metadata = model_metadata(
        model, RERANKER_MODEL_NAME, RERANKER_DECLARED_REVISION, RERANKER_MODEL_PATH
    )
    model.to("cuda")
    model.eval()
    return {
        "tokenizer": tokenizer,
        "model": model,
        "metadata": metadata,
        "load_seconds": perf_counter() - started,
    }


def score_pairs(reranker: dict, pairs: list[tuple[str, str]]) -> dict:
    scores = []
    torch.cuda.synchronize()
    started = perf_counter()
    for batch_start in range(0, len(pairs), RERANKER_BATCH_SIZE):
        batch = pairs[batch_start:batch_start + RERANKER_BATCH_SIZE]
        inputs = reranker["tokenizer"](
            [question for question, _ in batch],
            [chunk for _, chunk in batch],
            padding=True,
            truncation="only_second",
            max_length=RERANKER_MAX_SEQUENCE_LENGTH,
            return_tensors="pt",
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        with torch.no_grad():
            logits = reranker["model"](
                **inputs, return_dict=True
            ).logits.view(-1).float()
        values = logits.cpu().tolist()
        if len(values) != len(batch) or not all(isfinite(value) for value in values):
            raise RuntimeError("reranker returned invalid scores")
        scores.extend(float(value) for value in values)
    torch.cuda.synchronize()
    return {
        "scores": scores,
        "pairs": len(pairs),
        "seconds": perf_counter() - started,
    }


def rerank_candidates(reranker: dict, samples: dict, candidates: dict, chunks: list[dict]) -> dict:
    pairs = []
    layout = []
    for sample_id, sample in samples.items():
        support_counts = []
        for candidate in candidates[sample_id]:
            indices = candidate["supporting_chunk_indices"]
            if not 1 <= len(indices) <= SUPPORT_POOL_SIZE:
                raise RuntimeError("candidate must have between one and eight dense supporting chunks")
            support_counts.append(len(indices))
            pairs.extend((sample["question"], chunks[index]["text"]) for index in indices)
        layout.append((sample_id, support_counts))

    scoring = score_pairs(reranker, pairs)
    offset = 0
    rankings = {}
    for sample_id, support_counts in layout:
        reranked = []
        for candidate, count in zip(candidates[sample_id], support_counts):
            chunk_scores = scoring["scores"][offset:offset + count]
            offset += count
            if len(chunk_scores) != count:
                raise RuntimeError("cross-encoder score layout mismatch")
            selected_chunk_scores = sorted(chunk_scores, reverse=True)[:CE_SELECTED_CHUNKS]
            reranked.append({
                "document_id": candidate["document_id"],
                "cross_encoder_score": sum(selected_chunk_scores),
                "original_dense_rank": candidate["original_rank"],
            })
        reranked.sort(key=lambda document: (
            -document["cross_encoder_score"],
            document["original_dense_rank"],
            document["document_id"],
        ))
        ranking = [document["document_id"] for document in reranked]
        dense_ids = [document["document_id"] for document in candidates[sample_id]]
        if len(ranking) != len(set(ranking)) or set(ranking) != set(dense_ids):
            raise RuntimeError("cross-encoder changed the dense candidate universe")
        rankings[sample_id] = ranking
    if offset != len(scoring["scores"]):
        raise RuntimeError("not every cross-encoder score was consumed")
    return {"rankings": rankings, "scoring": scoring}


def build_predictions(public_samples: dict, rankings: dict) -> dict:
    predictions = {}
    for sample_id in public_samples:
        ranking = rankings.get(sample_id)
        if ranking is None:
            raise KeyError(f"sample {sample_id!r}: missing final ranking")
        answer = ranking[:FINAL_K]
        if len(answer) != FINAL_K or len(answer) != len(set(answer)):
            raise ValueError(f"sample {sample_id!r}: expected five unique document IDs")
        if not all(isinstance(document_id, str) for document_id in answer):
            raise TypeError(f"sample {sample_id!r}: document IDs must already be strings")
        predictions[sample_id] = {"answer": answer}
    return predictions


def validate_predictions(predictions: dict, public_samples: dict, corpus_ids: set[str]) -> dict:
    input_ids = set(public_samples)
    prediction_ids = set(predictions)
    missing_count = len(input_ids - prediction_ids)
    extra_count = len(prediction_ids - input_ids)
    duplicate_answer_count = 0
    answer_lengths = defaultdict(int)
    all_document_ids_strings = True

    if len(public_samples) != EXPECTED_PUBLIC_SAMPLES:
        raise ValueError("public input sample count changed after validation")
    if len(predictions) != EXPECTED_PUBLIC_SAMPLES:
        raise ValueError(f"expected exactly {EXPECTED_PUBLIC_SAMPLES} predictions")
    for sample_id, value in predictions.items():
        if not isinstance(value, dict) or set(value) != {"answer"}:
            raise ValueError(f"sample {sample_id!r}: output must contain answer only")
        answer = value["answer"]
        if not isinstance(answer, list):
            raise TypeError(f"sample {sample_id!r}: answer must be a list")
        answer_lengths[len(answer)] += 1
        if len(answer) != FINAL_K:
            raise ValueError(f"sample {sample_id!r}: answer must contain exactly five IDs")
        if len(answer) != len(set(answer)):
            duplicate_answer_count += 1
        if not all(isinstance(document_id, str) for document_id in answer):
            all_document_ids_strings = False
        if any(document_id not in corpus_ids for document_id in answer):
            raise ValueError(f"sample {sample_id!r}: answer contains an ID absent from corpus")

    if missing_count or extra_count:
        raise ValueError(
            f"prediction IDs mismatch: missing count={missing_count}, extra count={extra_count}"
        )
    if duplicate_answer_count:
        raise ValueError(
            f"duplicate document IDs found in {duplicate_answer_count} answers"
        )
    if not all_document_ids_strings:
        raise TypeError("every predicted document ID must be a JSON string")
    return {
        "input_samples": len(public_samples),
        "prediction_samples": len(predictions),
        "missing_id_count": missing_count,
        "extra_id_count": extra_count,
        "duplicate_answer_count": duplicate_answer_count,
        "answer_length_distribution": {
            str(length): answer_lengths[length] for length in sorted(answer_lengths)
        },
        "all_document_ids_strings": all_document_ids_strings,
        "all_document_ids_in_corpus": True,
        "expected_ids_exact_match": True,
    }


In [ ]:
# Run once on offline Kaggle; this cell writes prediction and audit files separately.
if not torch.cuda.is_available():
    raise RuntimeError("This inference notebook requires a Kaggle CUDA accelerator")

run_started = perf_counter()
torch.cuda.reset_peak_memory_stats()
public_samples = load_public_legalir(PUBLIC_LEGALIR_PATH)
documents = load_corpus(CORPUS_PATH)
chunks = chunk_corpus(documents)
if len(documents) != EXPECTED_DOCUMENTS or len(chunks) != EXPECTED_CHUNKS:
    raise ValueError(
        f"expected {EXPECTED_DOCUMENTS:,} documents / {EXPECTED_CHUNKS:,} chunks, "
        f"got {len(documents):,} / {len(chunks):,}"
    )
corpus_ids = {document["document_id"] for document in documents}
if len(corpus_ids) != EXPECTED_DOCUMENTS:
    raise RuntimeError("corpus ID cardinality mismatch")

dense = load_dense_model()
dense_metadata = dense["metadata"]
dense_load_seconds = dense["load_seconds"]
corpus_encoding = encode_normalized_cls(
    dense, [chunk["text"] for chunk in chunks], CORPUS_BATCH_SIZE
)
query_encoding = encode_normalized_cls(
    dense, [sample["question"] for sample in public_samples.values()], QUERY_BATCH_SIZE
)
dense_retrieval = retrieve_dense(
    query_encoding["embeddings"],
    corpus_encoding["embeddings"],
    chunks,
    list(public_samples),
)

dense["model"].to("cpu")
del dense, corpus_encoding["embeddings"], query_encoding["embeddings"]
gc.collect()
torch.cuda.empty_cache()

reranker = load_reranker()
reranked = rerank_candidates(
    reranker, public_samples, dense_retrieval["candidates"], chunks
)
predictions = build_predictions(public_samples, reranked["rankings"])
validation = validate_predictions(predictions, public_samples, corpus_ids)

assert set(predictions) == set(public_samples)
assert len(predictions) == EXPECTED_PUBLIC_SAMPLES
assert validation["missing_id_count"] == 0
assert validation["extra_id_count"] == 0
assert validation["duplicate_answer_count"] == 0
assert validation["answer_length_distribution"] == {str(FINAL_K): EXPECTED_PUBLIC_SAMPLES}
assert validation["all_document_ids_strings"]
assert validation["all_document_ids_in_corpus"]

metadata = {
    "task": "LegalIR public inference",
    "method": "current public-score reference: fixed 2000/200 + dense sum-top2 + candidate depth100 + m8 + CE select top2/sum-top2 + top5",
    "public_reference": "Dense-to-CE m8 / candidate100",
    "best_observed_public_score": 0.8935,
    "depth100_fixed_local_holdout": {
        "precision": 0.1884652981427175,
        "recall": 0.8888074291300098,
        "mrr": 0.7582601128237376,
        "ce_pairs": 619943,
    },
    "candidate50_status": "fixed-local-holdout-validated efficiency alternative; not current public-score reference",
    "candidate50_public_score": 0.8915,
    "input_path": str(PUBLIC_LEGALIR_PATH),
    "output_path": str(OUTPUT_PATH),
    "metadata_path": str(METADATA_PATH),
    "corpus": {
        "documents": len(documents),
        "chunks": len(chunks),
        "representation": "fixed character windows",
        "chunk_size": CHUNK_SIZE,
        "overlap": CHUNK_OVERLAP,
        "title_enrichment": False,
        "article_aware": False,
    },
    "dense_retrieval": {
        "model": dense_metadata,
        "embedding": "L2-normalized CLS last hidden state",
        "similarity": "dot product",
        "query_instruction": None,
        "max_length": DENSE_MAX_LENGTH,
        "dtype": "float16",
        "corpus_batch_size": CORPUS_BATCH_SIZE,
        "query_batch_size": QUERY_BATCH_SIZE,
        "top_k_chunks": TOP_K_CHUNKS,
        "document_aggregation": "sum top-2 dense chunk scores",
        "candidate_depth": CANDIDATE_DEPTH,
        "supporting_chunks": "up to 8 highest-dense-scoring occurrences from the original global top-2000 pool",
    },
    "cross_encoder": {
        "model": reranker["metadata"],
        "max_sequence_length": RERANKER_MAX_SEQUENCE_LENGTH,
        "dtype": "float16",
        "batch_size": RERANKER_BATCH_SIZE,
        "pair_scope": "question and one dense supporting chunk",
        "selected_chunks": "top 2 by independent CE score",
        "document_score": "sum of top-2 CE chunk scores",
        "tie_break": "CE score desc, original dense rank asc, document_id asc",
    },
    "final_k": FINAL_K,
    "validation": validation,
    "runtime": {
        "dense_model_load_seconds": dense_load_seconds,
        "corpus_encoding_seconds": corpus_encoding["seconds"],
        "query_encoding_seconds": query_encoding["seconds"],
        "dense_retrieval_seconds": dense_retrieval["seconds"],
        "reranker_load_seconds": reranker["load_seconds"],
        "reranking_seconds": reranked["scoring"]["seconds"],
        "cross_encoder_pairs": reranked["scoring"]["pairs"],
        "total_seconds": perf_counter() - run_started,
        "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated()),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
    },
}

OUTPUT_PATH.write_text(
    json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8"
)
METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(
    "Prediction file:\n"
    f"  {OUTPUT_PATH}\n\n"
    "Audit metadata:\n"
    f"  {METADATA_PATH}\n\n"
    "For Codabench:\n"
    "  package ONLY submission.json"
)
